# Single-sugar Zenteno model
Copy of diagnostic 17 with the S model and the G/F reference. Kie0 is fixed to the value from diagnostic 16. Only the five remaining chemical parameters are fitted, using LAB004–009 and the original two deterministic starting points and bounds.
No alternative S trajectories are calculated. Gas parameters, chemical equations, YAN treatment, weights, corrected initial biomass, instantaneous nutrient pulses and numerical tolerances are unchanged.
Figures use English labels, internal legends, compact spacing and per-LAB RMSE. Initial points are shown with the samples; initial values are excluded from the chemical RMSE. Historical gas training: LAB019–021. Gas holdout: LAB016–018. Chemical holdout does not imply joint blind validation of the gas model.


## Ecuaciones y alcance
\[
\mu=\mu_0a_\mu(T)\frac{N}{N+K_N(T)},\qquad
\beta_S=\beta_{S0}a_\beta(T)\frac{S}{S+K_S(T)}\frac{K_{iE}(T)}{E+K_{iE}(T)}.
\]
\[
\dot X=(\mu-k_d)X,\quad \dot N=-\mu X/Y_{X/N},\quad
\dot S=-X[\mu/Y_{X/S}+\beta_S/Y_{E/S}+m(T)S/(S+1)],\quad \dot E=\beta_SX.
\]
Concentraciones en g/L, tiempo en horas; el 1 del factor de mantenimiento es 1 g/L, igual al módulo original. Se anula mu si S está agotado. Nutrición: salto instantáneo de N, muestra previa al salto si coincide.
\[
qprod=(\mu/12.72+\beta_S/1.05)X,\quad
\dot C_d=\min(A\,qprod,k_La\max(C_{sat}-C_d,0)),\quad
q_{gas}=A\,qprod-\dot C_d,\quad q_{pred}=g\,q_{gas}.
\]
A usa la misma transición suave 5–10 g/L de azúcar consumido que el 16. La mortalidad conserva exactamente su umbral y ley térmica/etanol. K_N y K_S conservan la ley térmica correspondiente del original.

## Comparison
Two models: G/F from diagnostic 16 and the single-sugar S model. Density is a graphical comparison, not a fitting target. Initial chemical values are excluded from RMSE. The fitting and scoring supports are unchanged from diagnostic 17.


In [ ]:
from pathlib import Path
from dataclasses import replace
import sys,json,hashlib,math,time
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares,minimize_scalar
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/wine_fermentation').is_dir())
assert Path(sys.prefix).name.lower()=='fermentation'
sys.path.insert(0,str(ROOT/'src'));sys.dont_write_bytecode=True
from wine_fermentation import model as kin
from wine_fermentation.data import natural_batches
from wine_fermentation.models.historical.calibration import load
from wine_fermentation.comparison import observations,chemical_observations,CHEMICAL_COLUMNS
NAME='19_zenteno_single_sugar_clean_figures'
protected=[p for folder in ['src','configs','data/compatibility/chemistry','data/cleaned/natural/lab004_009','notebooks'] for p in (ROOT/folder).rglob('*') if p.is_file() and '__pycache__' not in str(p) and not p.name.startswith(NAME+'.')]
hashes={p:hashlib.sha256(p.read_bytes()).hexdigest() for p in protected}
data,training,profiles=natural_batches(ROOT)
# Correcciones de recuento inicial indicadas por el usuario; originales conservados.
INITIAL_VIABLE_MCELLS_ML={'LAB004':10.75,'LAB005':11.25}
initial_biomass_audit=[];corrected_training=[]
for batch in training:
 if batch.batch in INITIAL_VIABLE_MCELLS_ML:
  count=INITIAL_VIABLE_MCELLS_ML[batch.batch];x0=count*.03
  ids=np.flatnonzero(np.isclose(batch.time,0.));assert len(ids)==1
  obs0={k:v.copy() for k,v in batch.observations.items()};obs0['X'][ids[0]]=x0
  initial_biomass_audit.append(dict(LAB=batch.batch,original_g_L=batch.initials['X'],viables_millones_mL=count,corregido_g_L=x0))
  batch=replace(batch,initials={**batch.initials,'X':x0},observations=obs0)
  mask=data.batch.eq(batch.batch)&np.isclose(data.time_h,0.)
  assert mask.sum()==1
  data.loc[mask,'X_viable_kg_m3']=x0;data.loc[mask,'X_viable_mcells_ml']=count
  assert batch.initials['X']==batch.observations['X'][ids[0]]
 corrected_training.append(batch)
training=corrected_training
assert len(initial_biomass_audit)==2
display(pd.DataFrame(initial_biomass_audit))

start=kin.normalize_parameters(json.loads((ROOT/'configs/theta/theta_full.json').read_text()))
config=json.loads((ROOT/'configs/models/without_transporters/calibration.json').read_text())
# Tabla 2 de Zenteno: Kn=0.01 g/L a23C; nuestro Kn0 se referencia a20C.
KN_ZENTENO_G_L=0.01;KN_PAPER_REF_K=296.15;KN_CODE_REF_K=293.15
KN_THERMAL_FACTOR=np.exp(kin.FIXED_CONSTANTS['EaKn']*(KN_PAPER_REF_K-KN_CODE_REF_K)/(KN_CODE_REF_K*kin.FIXED_CONSTANTS['R']*KN_PAPER_REF_K))
start['Kn0']=KN_ZENTENO_G_L/KN_THERMAL_FACTOR
KD0_ZENTENO_H=0.00044
start['Kd0']=KD0_ZENTENO_H
assert np.isclose(start['Kn0']*KN_THERMAL_FACTOR,KN_ZENTENO_G_L,rtol=1e-14)
display(pd.Series(dict(Kn_Zenteno_g_L=KN_ZENTENO_G_L,referencia_paper_C=23,Kn0_codigo_g_L=start['Kn0'],referencia_codigo_C=20),name='Kn fijo y conversión de referencia térmica'))
names=[k for k in config['bounds'] if k not in ['aN','KNa','Kn0','Kd0']]
assert len(names)==8 and 'Kn0' not in names and 'Kd0' not in names
bounds={k:config['bounds'][k] for k in names};lo,hi=np.array(list(bounds.values())).T
fixed={k:v for k,v in start.items() if k not in names}
display(pd.Series(fixed,name='Fijos: theta_full salvo Kn y Kd0 de Zenteno'))
display(pd.DataFrame([dict(parametro=k,inicio=start[k],inicio_factible=np.clip(start[k],lo[i]+1e-10,hi[i]-1e-10),minimo=lo[i],maximo=hi[i]) for i,k in enumerate(names)]))


diffs=[]
bylab={b.batch:b for b in training}
for a,b in [('LAB004','LAB005'),('LAB007','LAB008')]:
 aa=pd.Series(bylab[a].observations['X'],index=bylab[a].time);bb=pd.Series(bylab[b].observations['X'],index=bylab[b].time)
 z=pd.concat([aa,bb],axis=1).dropna();diffs.extend((z.iloc[:,0]-z.iloc[:,1]).tolist())
sigma_rep=float(np.sqrt(np.mean(np.square(diffs)))/np.sqrt(2));SIGMA_X=max(.5,3*sigma_rep)
assert len(diffs)>0
print('Pares de réplicas:',len(diffs),'sigma réplica:',sigma_rep,'sigmaX aplicada:',SIGMA_X)
treated=[];audit=[];rules=[]
for b in training:
 obs={k:v.copy() for k,v in b.observations.items()};old=obs['N'].copy()
 obs['N'][np.isfinite(old)&(old<=.020)]=0
 if b.batch=='LAB006':
  ids=np.flatnonzero(np.isclose(b.time,122)&np.isfinite(old));assert len(ids)==1 and np.isclose(old[ids[0]],.027)
  obs['N'][ids]=0;depletion=122.
 else:
  last=max(t for t,d in b.pulses['N']);ids=np.flatnonzero((b.time>=last)&np.isfinite(obs['N'])&(obs['N']==0));assert len(ids)
  depletion=float(b.time[ids[0]])
 obs['N'][b.time>depletion+12]=np.nan
 for i in np.flatnonzero(np.isfinite(old)):
  audit.append(dict(LAB=b.batch,t_h=b.time[i],original_mg=old[i]*1000,usado_mg=obs['N'][i]*1000,estado='Inicial' if b.time[i]==0 else ('Excluido' if not np.isfinite(obs['N'][i]) else ('Transformado' if old[i]!=obs['N'][i] else 'Usado'))))
 assert all(np.array_equal(obs[k],b.observations[k],equal_nan=True) for k in obs if k!='N')
 treated.append(replace(b,observations=obs))
 rules.append(dict(LAB=b.batch,agotamiento_h=depletion,corte_YAN_h=depletion+12,N0_mg=b.initials['N']*1000,pulsos=b.pulses['N'],**{k:int(np.count_nonzero(np.isfinite(obs[k])&(b.time>0))) for k in obs}))
audit=pd.DataFrame(audit);display(pd.DataFrame(rules));display(audit)
def sigma(state,y):return np.full_like(y,SIGMA_X) if state=='X' else kin.sigma_for_state(state,y)
def simulate(b,p):
 def rhs(t,y):
  out=np.asarray(kin.rhs(t,y,p,b))
  if y[2]<=0 or y[3]<=0:
   q=kin.kinetic_terms(p,kin.temperature_at(b,t),*y);growth=q['mu']*max(float(y[0]),0)
   out[0]-=growth;out[1]+=growth/p['Yxn'];out[2]+=growth/p['Yxg'];out[3]+=growth/p['Yxf']
  return out
 # Muestra previa al salto si coincide con la nutrición; dosis completa después.
 minimum=np.full(5,np.inf);jumps=[];events={}
 for tp,dose in b.pulses.get('N',()):
  assert tp>float(b.time[0]) and dose>=0,'No se admiten pulsos iniciales extra'
  if tp<=b.time[-1]:events[float(tp)]=events.get(float(tp),0.)+float(dose)
 times=np.asarray(b.time);state=kin.initial_vector(b);records={float(times[0]):state.copy()};now=float(times[0])
 for end in sorted(set([float(times[-1])]+list(events))):
  requested=times[(times>now)&(times<=end)]
  sol=solve_ivp(rhs,(now,end),state,t_eval=np.unique(np.r_[requested,end]),method='LSODA',rtol=2e-6,atol=2e-8,max_step=.5)
  if not sol.success or not np.isfinite(sol.y).all() or np.any(sol.y.min(axis=1)<-np.array([1e-6,1e-6,1e-4,1e-4,1e-6])):raise RuntimeError('Integración química inválida '+b.batch+' '+str(sol.message)+' mínimos='+str(dict(zip(kin.STATE_NAMES,sol.y.min(axis=1)))))
  minimum=np.minimum(minimum,sol.y.min(axis=1))
  records.update({float(t):y.copy() for t,y in zip(sol.t,sol.y.T)});state=sol.y[:,-1].copy();now=end
  if end in events:
   before=state.copy();state[1]+=events[end]
   assert np.isclose(state[1]-before[1],events[end],atol=1e-12) and np.array_equal(state[[0,2,3,4]],before[[0,2,3,4]])
   jumps.append(dict(t=end,dose=events[end],before=before.copy(),after=state.copy()))
 f=pd.DataFrame(np.maximum(np.vstack([records[float(t)] for t in times]),0),index=times,columns=kin.STATE_NAMES);f['GF']=f.G+f.F
 f.attrs['minimum_integrated_states']=dict(zip(kin.STATE_NAMES,minimum.tolist()))
 f.attrs['pulse_jumps']=jumps
 assert np.isclose(sum(j['dose'] for j in jumps),sum(events.values()),atol=1e-12)
 return f

def with_jumps(f):
 extra=[];tt=[]
 for j in f.attrs['pulse_jumps']:
  if j['t'] not in f.index:extra.append(j['before']);tt.append(j['t'])
  extra.append(j['after']);tt.append(j['t'])
 if not extra:return f
 added=pd.DataFrame(extra,index=tt,columns=kin.STATE_NAMES);added['GF']=added.G+added.F
 return pd.concat([f,added]).sort_index(kind='stable')

support=[(b,s,np.isfinite(b.observations[s])&(b.time>0)) for b in treated for s in kin.STATE_NAMES]
NR=sum(m.sum() for b,s,m in support);calls=0;rejected=0


# Química congelada de la recalibración anterior; sin optimización en esta prueba.
theta={'mu0': 0.09403147367985473, 'betaG0': 0.42243676903335664, 'betaF0': 0.6236552480900203, 'Kn0': 0.008257860224319971, 'Kg0': 4.400775419123265, 'Kf0': 19.296083508118652, 'Kig0': 65.01993873457147, 'Kie0': 96.26516563486751, 'Kd0': 0.00044, 'Yxn': 6.456225797038313, 'Yxg': 0.9660935553477866, 'Yxf': 1.1098958529485594, 'Yeg': 0.4684585383875136, 'Yef': 0.5113278649826252, 'm0': 0.0183385793928771}
assert theta['Kn0']==start['Kn0'] and theta['Kd0']==KD0_ZENTENO_H
assert all(theta[k]==v for k,v in fixed.items())
print('PARÁMETROS QUÍMICOS COMPLETOS',json.dumps(theta,indent=2))
print('Química fija: no se recalibra en esta prueba de activación.')
traincurves={b.batch:simulate(b,theta) for b in treated}


v=load(ROOT);LABS=['LAB019','LAB020','LAB021','LAB016','LAB017','LAB018'];TRAIN=LABS[:3];HOLD=LABS[3:]
raw=pd.read_csv(ROOT/'data/processed/co2_mass_balance_lab016_021_g_l_h.csv',usecols=['lab','time_h','record_kind','remaining_volume_L','co2_raw_g_l_h']);obs=observations(raw)
batches={};cores={};volumes={};caches={};activation={};entry=[]
for lab in LABS:
 b=v.batches[lab];ev=raw.loc[raw.lab.eq(lab)&raw.record_kind.eq('withdrawal_after')].sort_values('time_h');et=ev.time_h.to_numpy();vv=ev.remaining_volume_L.to_numpy()
 def volume(t,et=et,vv=vv):return np.r_[2.,vv][np.searchsorted(et,np.asarray(t),side='right')]
 volumes[lab]=volume
 if lab in TRAIN:b=replace(b,pulses={**b.pulses,'N':tuple((t,d*2/float(volume(t))) for t,d in b.pulses['N'])})
 if lab=='LAB017':b=replace(b,pulses={**b.pulses,'N':((min(t for t,d in b.pulses['N']),sum(d for t,d in b.pulses['N'])),)})
 if lab=='LAB021':b=replace(b,initials={**b.initials,'X':9.*.03})
 batches[lab]=b;f=simulate(b,theta);cores[lab]=f;f=with_jumps(f)
 tt=np.unique(np.r_[np.arange(0,float(f.index[-1]),.1),f.index[-1]])
 rates=[]
 for t in tt:
  y=[np.interp(t,f.index,f[s]) for s in kin.STATE_NAMES];q=kin.kinetic_terms(theta,kin.temperature_at(b,t),*y)
  if y[2]<=0 or y[3]<=0:q['mu']=0.
  rates.append((q['mu']/12.72+(q['beta_g']+q['beta_f'])/1.05)*y[0])
 caches[lab]=(tt,np.asarray(rates))
 consumed=b.initials['G']+b.initials['F']-np.interp(tt,f.index,f.GF)
 z=np.clip((consumed-5.)/5.,0.,1.);activation[lab]=z*z*(3.-2.*z)
 assert np.all(activation[lab][consumed<=5.]==0) and np.all(activation[lab][consumed>=10.]==1)
 entry.append(dict(LAB=lab,**b.initials,pulsos=b.pulses['N'],corte_h=v.horizons[lab]))
 obs[lab]=obs[lab].loc[obs[lab].time_h.between(0,v.horizons[lab])].copy()
display(pd.DataFrame(entry))
CSAT=1.5
def gas(lab,k,gain=1,activated=True):
 t,source=caches[lab]
 if activated:source=source*activation[lab]
 cd=np.zeros(len(t));qout=np.zeros(len(t));acc=np.zeros(len(t));qout[0]=max(source[0]-k*CSAT,0)
 for i,dt in enumerate(np.diff(t),1):
  qp=(source[i-1]+source[i])/2;D=max(CSAT-cd[i-1],0)
  if qp<=0 or D<=0:delta=0.
  elif k*D<=qp:delta=D*(-np.expm1(-k*dt))
  else:
   crossing=(D-qp/k)/qp;linear=min(dt,crossing);delta=qp*linear
   if crossing<dt:delta+=(qp/k)*(-np.expm1(-k*(dt-crossing)))
  cd[i]=cd[i-1]+delta;acc[i]=delta/dt;qout[i]=qp-acc[i]
 assert cd.min()>=0 and cd.max()<=CSAT+1e-10 and qout.min()>-1e-10
 assert np.max(abs(np.diff(cd)-np.diff(t)*((source[:-1]+source[1:])/2-qout[1:])))<1e-10
 return pd.DataFrame(dict(qprod=source,Cd=cd,qgas=qout,qpred=gain*qout),index=t)
# Pareja calibrada originalmente; activación5–10g/L, sin reajuste.
KLA=0.07467887635557982;GAIN=0.635238911849421
print('PARÁMETROS GAS',json.dumps(dict(kLa=KLA,gain=GAIN,Csat=CSAT,Cd0=0,training=TRAIN,holdout=HOLD,fit=False),indent=2))
curves={lab:gas(lab,KLA,GAIN) for lab in LABS};metrics=[]
for lab in LABS:
 o=obs[lab];f=curves[lab];e=np.interp(o.time_h,f.index,f.qpred)-o.co2_raw_g_l_h
 metrics.append(dict(LAB=lab,rol='Calibración gas' if lab in TRAIN else 'Holdout gas',n=len(o),RMSE=np.sqrt(np.mean(e*e)),sesgo=np.mean(e)))
metrics=pd.DataFrame(metrics);display(metrics);display(metrics.groupby('rol')[['RMSE','sesgo']].mean())

display(pd.DataFrame([dict(LAB=lab,inicio_5g_h=float(caches[lab][0][np.flatnonzero(activation[lab]>0)[0]]),completa_10g_h=float(caches[lab][0][np.flatnonzero(activation[lab]>=1)[0]])) for lab in LABS]))


In [ ]:

# Referencia original guardada antes de construir el modelo reducido.
original_theta=theta.copy();original_train=traincurves.copy();original_cores=cores.copy();original_gas=curves.copy()
YXS=1/(1/theta['Yxg']+1/theta['Yxf']);STATES=['X','N','S','E']
PARAMS=['mu0','betaS0','Ks0','Kie0','Yxn','Yes']
LOW=np.array([.05,.1,.1,5.,1.,.1]);HIGH=np.array([1.,20.,200.,1000.,100.,.5114])
pinit=np.array([theta['mu0'],theta['betaG0']+theta['betaF0'],10.,theta['Kie0'],theta['Yxn'],.49])
def terms_s(t,y,p,b):
 x,n,s,e=np.maximum(y,0);tk=kin.temperature_at(b,t)+273.15;R=kin.FIXED_CONSTANTS['R']
 ak=np.exp(46055*(tk-293.15)/(293.15*R*tk))
 mu=p['mu0']*np.exp(59453*(tk-300)/(300*R*tk))*n/(n+theta['Kn0']*ak+1e-8) if s>0 else 0.
 beta=p['betaS0']*np.exp(11000*(tk-296.15)/(296.15*R*tk))*s/(s+p['Ks0']*ak+1e-8)*(p['Kie0']*(ak+1e-8)/(e+p['Kie0']*(ak+1e-8)))
 td=-.0001*e**3+.0049*e**2-.1279*e+315.89
 kd=theta['Kd0']*np.exp(.0415*e+130000*(tk-305.65)/(305.65*R*tk)) if tk>=td else 0.
 maint=theta['m0']*np.exp(37681*(tk-293.3)/(293.3*R*tk))*s/(s+kin.MAINTENANCE_SUGAR_CUTOFF_KG_M3)
 return mu,beta,kd,maint

def simulate_s(b,p):
 def rhs(t,y):
  x=max(y[0],0);mu,be,kd,m=terms_s(t,y,p,b)
  return [(mu-kd)*x,-mu*x/p['Yxn'],-(mu/YXS+be/p['Yes']+m)*x,be*x]
 tt=np.asarray(b.time);y=np.array([b.initials['X'],b.initials['N'],b.initials['G']+b.initials['F'],b.initials['E']]);now=tt[0];rec={now:y.copy()};events={};jumps=[];minimum=y.copy()
 for t,d in b.pulses.get('N',()):
  assert t>now and d>=0
  if t<=tt[-1]:events[t]=events.get(t,0)+d
 for end in sorted(set([float(tt[-1])]+list(events))):
  requested=np.unique(np.r_[tt[(tt>now)&(tt<=end)],end])
  sol=solve_ivp(rhs,(now,end),y,t_eval=requested,method='LSODA',rtol=2e-6,atol=2e-8,max_step=.5)
  if not sol.success or not np.isfinite(sol.y).all() or np.any(sol.y.min(axis=1)<-np.array([1e-6,1e-6,1e-4,1e-6])):raise RuntimeError('Integración inválida '+b.batch)
  minimum=np.minimum(minimum,sol.y.min(axis=1));rec.update({float(t):z.copy() for t,z in zip(sol.t,sol.y.T)})
  y=sol.y[:,-1].copy();now=end
  if end in events:
   before=y.copy();y[1]+=events[end];jumps.append((end,before,y.copy()))
 f=pd.DataFrame(np.maximum([rec[float(t)] for t in tt],0),index=tt,columns=STATES)
 f.attrs['jumps']=jumps;f.attrs['minima']=minimum;return f

def plotframe_s(f):
 rows=[];times=[]
 for t,before,after in f.attrs['jumps']:
  rows.extend([before,after]);times.extend([t,t])
 return pd.concat([f,pd.DataFrame(rows,index=times,columns=STATES)]).sort_index(kind='stable') if rows else f

def targets(b,s):
 y=b.observations[s] if s!='S' else b.observations['G']+b.observations['F']
 sig=sigma(s,y) if s!='S' else np.hypot(kin.sigma_for_state('G',b.observations['G']),kin.sigma_for_state('F',b.observations['F']))
 return y,sig,np.isfinite(y)&(b.time>0)
support_s=[(b,s,*targets(b,s)) for b in treated for s in STATES]
NR_S=sum(mask.sum() for b,s,y,sg,mask in support_s);rejections=0
print('Soporte de ajuste');display(pd.DataFrame([dict(LAB=b.batch,estado=s,n=int(mask.sum())) for b,s,y,sg,mask in support_s]))
def residual_s(logp):
 global rejections
 p=dict(zip(PARAMS,np.exp(logp)))
 try:
  predictions={b.batch:simulate_s(b,p) for b in treated}
  return np.concatenate([(predictions[b.batch][s].to_numpy()[mask]-y[mask])/sg[mask] for b,s,y,sg,mask in support_s])
 except RuntimeError:
  rejections+=1;return np.full(NR_S,1e5)
FIT_SEED=np.array([0.08934685518404714,0.7790518060881909,51.79327570452845,6.193688479843567,0.4893196389269888])
FIT_PARAMS=[k for k in PARAMS if k!='Kie0'];SELECT=[PARAMS.index(k) for k in FIT_PARAMS];KIE_FIXED=original_theta['Kie0']
LOW_FIT=LOW[SELECT];HIGH_FIT=HIGH[SELECT]
def expand_fixed(logp):
 p=dict(zip(FIT_PARAMS,np.exp(logp)));p['Kie0']=KIE_FIXED
 return np.log([p[k] for k in PARAMS])
def residual_fixed(logp):return residual_s(expand_fixed(logp))
fits=[]
for i,mult in enumerate([np.ones(5),np.array([1.1,1.2,.5,.9,1.])]):
 seed=np.clip(FIT_SEED*mult,LOW_FIT*1.00001,HIGH_FIT*.99999)
 result=least_squares(residual_fixed,np.log(seed),bounds=(np.log(LOW_FIT),np.log(HIGH_FIT)),diff_step=1e-3,max_nfev=180,ftol=1e-6,xtol=1e-6,gtol=1e-5)
 fits.append(result);print('Inicio Kie fijo',i,'MSE',np.mean(result.fun**2),'nfev',result.nfev,'convergencia',result.success,result.message,flush=True)
fit=min(fits,key=lambda r:np.sum(r.fun**2));ps=dict(zip(PARAMS,np.exp(expand_fixed(fit.x))))
ps['Kie0']=KIE_FIXED
print('PARÁMETROS S',json.dumps(ps,indent=2));print('Fijos',dict(Kn0=theta['Kn0'],Kd0=theta['Kd0'],Yxs=YXS,m0=theta['m0'],kLa=KLA,gain=GAIN),'rechazos',rejections)
display(pd.DataFrame(dict(parametro=FIT_PARAMS,valor=np.exp(fit.x),minimo=LOW_FIT,maximo=HIGH_FIT,cerca_cota=(np.minimum(fit.x-np.log(LOW_FIT),np.log(HIGH_FIT)-fit.x)<.01))))
print('Valores singulares Jacobiano log-parámetros',np.linalg.svd(fit.jac,compute_uv=False))
reduced_train={b.batch:simulate_s(b,ps) for b in treated};reduced_cores={lab:simulate_s(b,ps) for lab,b in batches.items()}
# Reutiliza exactamente el integrador de gas del 16 con fuente del modelo reducido.
original_caches=caches;original_activation=activation
caches={};activation={}
for lab,f in reduced_cores.items():
 b=batches[lab];f=plotframe_s(f);tt=original_caches[lab][0]
 production=[]
 for t in tt:
  y=[np.interp(t,f.index,f[s]) for s in STATES];mu,be,_,_=terms_s(t,y,ps,b);production.append((mu/12.72+be/1.05)*y[0])
 caches[lab]=(tt,np.array(production));z=np.clip((b.initials['G']+b.initials['F']-np.interp(tt,f.index,f.S)-5)/5,0,1);activation[lab]=z*z*(3-2*z)
reduced_gas={lab:gas(lab,KLA,GAIN) for lab in LABS}

assert ps['Kie0']==theta['Kie0']
display(pd.Series(ps,name='S model parameters'))


In [ ]:
comparison=[]

def original_state(f,s):return f['GF' if s=='S' else s]

def finish_figure(fig,axs,title):
 fig.suptitle(title,y=.99)
 fig.tight_layout(pad=.7,h_pad=1.2,w_pad=.9)
 for ax in axs.flat:
  handles,labels=ax.get_legend_handles_labels();unique=dict(zip(labels,handles))
  legend=ax.legend(unique.values(),unique.keys(),loc='best',fontsize=7.5,framealpha=.85,title=getattr(ax,'_rmse_text',None),title_fontsize=7.5)
  assert len(legend.get_texts())==len(unique)
 assert not fig.legends
 plt.show()

def draw_group(group,reference,reduced,holdout=False):
 for s,title in [('X','Biomass'),('N','YAN'),('S','Glucose + fructose'),('E','Ethanol')]:
  scale=1000 if s=='N' else 1;unit='mg/L' if s=='N' else 'g/L'
  fig,axs=plt.subplots(2,3,figsize=(15,8))
  for ax,item in zip(axs.flat,group):
   b=batches[item] if holdout else item;lab=b.batch;a=with_jumps(reference[lab]);r=plotframe_s(reduced[lab])
   ax.plot(a.index,original_state(a,s)*scale,c='gray',ls='--',label='G/F model')
   ax.plot(r.index,r[s]*scale,c='#0072B2',label='S model')
   label='G+F' if s=='S' else 'Samples'
   if not holdout:
    yy,sg,mask=targets(b,s);tt=b.time
    ax.scatter(tt[mask],yy[mask]*scale,c='black',s=23,label=label)
   else:
    col='GF' if s=='S' else s
    if lab in TRAIN:
     z=chemical_observations(v.chem,lab,col);tt=z.t_h.to_numpy();yy=z[CHEMICAL_COLUMNS[col]].to_numpy()
    elif s=='X':
     z=v.biomass.loc[v.biomass.lab.eq(lab)].dropna(subset=['t_h','X_viable_g_l']);tt=z.t_h.to_numpy();yy=z.X_viable_g_l.to_numpy()
    else:tt=np.array([]);yy=np.array([])
    mask=np.isfinite(yy)&(tt>0)&(tt<=v.horizons[lab]);shown=np.isfinite(yy)&~np.isclose(tt,0.)
    if shown.any():ax.scatter(tt[shown],yy[shown]*scale,c='black',s=23,label=label)
    if s=='S':
     z=v.density.loc[v.density.lab.eq(lab)].dropna(subset=['t_h','density_g_l'])
     ax.scatter(z.t_h,2.5*(z.density_g_l-1000),marker='D',facecolors='none',edgecolors='#D55E00',s=32,label='G+F from density')
   # Use the actual corrected model initial value, never the superseded biomass observation.
   if not (holdout and s=='S' and lab not in TRAIN):
    initial_point=ax.scatter([0.],[r[s].iloc[0]*scale],c='black',s=23,label=label)
    assert np.allclose(initial_point.get_offsets(),[[0.,r[s].iloc[0]*scale]])
   errors=[]
   for model,f,vals in [('G/F',a,original_state(a,s)),('S',r,r[s])]:
    if mask.any():
     # Same scoring operator and samples as diagnostic 17.
     err=(np.interp(tt[mask],f.index,vals)-yy[mask])*scale
     value=float(np.sqrt(np.mean(err**2)));errors.append(f'RMSE {model} = {value:.3f} {unit}')
     comparison.append(dict(LAB=lab,rol='Holdout' if holdout else 'Calibration',variable=s,modelo=model,n=int(mask.sum()),RMSE=value,sesgo=float(err.mean())))
   if holdout and s=='S' and not mask.any():
    # Label density-based errors explicitly; do not add them to the chemical scoring table.
    density_mask=z.t_h.gt(0)&z.t_h.le(v.horizons[lab]);dt=z.loc[density_mask,'t_h'].to_numpy();dy=2.5*(z.loc[density_mask,'density_g_l'].to_numpy()-1000)
    for model,f,vals in [('G/F',a,original_state(a,s)),('S',r,r[s])]:
     if len(dt):errors.append(f'RMSE {model} (density) = {np.sqrt(np.mean((np.interp(dt,f.index,vals)-dy)**2)):.3f} g/L')
   if errors:ax._rmse_text='\n'.join(errors)
   if s=='N':
    for t,d in b.pulses['N']:ax.axvline(t,c='gray',ls=':',lw=.8,label='N pulse')
   ax.set(title=lab,xlabel='Time [h]',ylabel=title+f' [{unit}]',ylim=(0,None));ax.grid(alpha=.2)
   if holdout and s=='S':
    ax.set_ylim(bottom=min(0.,float((2.5*(z.density_g_l-1000)).min())*1.15));ax.axhline(0,c='gray',lw=.6)
  finish_figure(fig,axs,title+' · '+('Holdout' if holdout else 'Calibration LAB004–009'))

draw_group(treated,original_train,reduced_train)
draw_group(LABS,original_cores,reduced_cores,True)
fig,axs=plt.subplots(2,3,figsize=(15,8))
for ax,lab in zip(axs.flat,LABS):
 o=obs[lab];ax.plot(o.time_h,o.co2_raw_g_l_h,c='black',lw=.8,label='Samples');errors=[]
 for model,cur,color,style in [('G/F',original_gas,'gray','--'),('S',reduced_gas,'#0072B2','-')]:
  f=cur[lab];ax.plot(f.index,f.qpred,c=color,ls=style,label=model+' model');err=np.interp(o.time_h,f.index,f.qpred)-o.co2_raw_g_l_h
  value=float(np.sqrt(np.mean(err**2)));errors.append(f'RMSE {model} = {value:.3f} g/(L h)')
  comparison.append(dict(LAB=lab,rol='Gas evaluation',variable='CO2',modelo=model,n=len(o),RMSE=value,sesgo=float(err.mean())))
 ax._rmse_text='\n'.join(errors)
 ax.set(title=lab+(' · calibration' if lab in TRAIN else ' · holdout'),xlabel='Time [h]',ylabel='CO₂ [g/(L h)]',ylim=(0,None));ax.grid(alpha=.2)
finish_figure(fig,axs,'CO₂ · calibration LAB019–021 and holdout LAB016–018')
metrics_s=pd.DataFrame(comparison);display(metrics_s.pivot(index=['rol','LAB','variable','n'],columns='modelo',values='RMSE'));print('METRICS_JSON',metrics_s.to_json(orient='records'))

timing=[]
for lab,f in {**reduced_train,**reduced_cores}.items():
 a=original_train.get(lab,original_cores.get(lab))
 for model,frame in [('G/F',a),('S',f)]:
  for state,threshold in [('S',2.),('N',.001)]:
   values=original_state(frame,state) if model=='G/F' else frame[state];lastpulse=max(t for t,d in (bylab.get(lab,batches.get(lab))).pulses['N']);ids=np.flatnonzero((frame.index>lastpulse)&(values<=threshold))
   timing.append(dict(LAB=lab,model=model,state=state,first_time_after_pulse_h=float(frame.index[ids[0]]) if len(ids) else np.nan,residual=float(values.iloc[-1])))
display(pd.DataFrame(timing))
assert set(metrics_s.modelo)=={'G/F','S'}
assert ps['Kie0']==original_theta['Kie0']
assert theta==original_theta and theta['Kd0']==.00044 and np.isclose(theta['Kn0']*KN_THERMAL_FACTOR,.01)
assert KLA==.07467887635557982 and GAIN==.635238911849421
for lab,f in reduced_cores.items():
 for t,bef,aft in f.attrs['jumps']:assert np.array_equal(bef[[0,2,3]],aft[[0,2,3]])
for p,h in hashes.items():assert hashlib.sha256(p.read_bytes()).hexdigest()==h,str(p)
print('Previous files unchanged. No fitting on holdout samples.')


## Scope
The S model fixes Kie0 to diagnostic 16 and fits mu0, betaS0, Ks0, Yxn and Yes. The preceding cells report convergence, fitted parameters, bounds, sensitivity, common-sample errors and depletion times for G/F and S.
Initial points remain visible; chemical RMSE excludes them. The YAN transformation and exclusion rules affect scoring exactly as in diagnostic 17. Excluded points and untransformed original values are not displayed. Density errors are identified separately in the applicable plots. No correlation coefficient is used.
